# 같은 855개 군집에서 예측 방법 비교

4분기의 대상 시간과 공간 단위를 고정한다. 한 방법은 기존 500m 군집 모델이 낸 예측을 600m 상위 군집별로 더하고, 다른 방법은 처음부터 600m 군집으로 학습해 직접 예측한다. 동일한 대여소 집합과 동일한 실제 건수에 대해 MAE·RMSE를 비교한다. 모든 점수 단위는 600m 군집 한 곳의 한 시간당 건수다.

모델 폴더에 포함된 기존 500m 모델의 4분기 가중치 8개를 사용한다. 가중치를 재학습하려면 기존 1,021개 군집 실험 노트북 3개를 순서대로 실행한다. 먼저 실루엣 군집 수요예측 노트북을 실행해 새 군집 데이터와 직접 예측 결과를 준비한다.

In [1]:
from pathlib import Path
import os
import subprocess
import sys
from tempfile import TemporaryDirectory
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "전처리/rental_aggregate.py").is_file())
OUT = ROOT / "모델"
stations = pd.read_csv(ROOT / "따릉이 대여소 비교/서울 전체 직선거리 500m 군집 배정.csv")
stations = stations.loc[stations["기준월"].eq("2024-12")].set_index("대여소번호").sort_index()
group = stations["완전연결 군집번호"]
ids = pd.Index(sorted(group.unique()), name="군집번호")
hours = pd.date_range("2023-01-01", "2025-01-01", freq="h", inclusive="left")
assert len(stations) == 2766 and group.index.is_unique and len(ids) == 1021
assert stations["완전연결 군집 내 최대거리(m)"].max() <= 500
plt.rcParams.update({"font.family": "Malgun Gothic" if os.name == "nt" else "sans-serif",
                     "axes.unicode_minus": False, "figure.dpi": 110})
print(f"{len(hours):,}시간 × {len(ids):,}개 군집 = {len(hours)*len(ids):,}개 군집·시간")

17,544시간 × 1,021개 군집 = 17,912,424개 군집·시간


## 월별 출발 건수를 군집에 배정

0분 또는 0m 대여를 제외하고 중복을 정리한 기존 전처리 집계를 사용한다. 원본 ZIP에서 다시 만들 때는 임시 폴더를 사용한다. 이미 만들어 둔 월별 집계 폴더가 있다면 환경변수 BIKE_AGGREGATE_CACHE로 지정할 수 있다. 매월 기존 서울 전체 합계와 50개 표본의 합계를 독립적으로 대조한다.

In [2]:
verified = pd.read_csv(ROOT / "전처리/월별 집계 검증.csv").set_index("월")
sample_ids = pd.read_csv(ROOT / "전처리/선정 대여소.csv").station_id
totals = np.zeros((len(hours), len(ids)), dtype=np.int32)
# 기준선에서 시차 대여량 중 예측 정시에 아직 반납이 확인되지 않은 건수를 뺀다.
late_returns = {lag: np.zeros_like(totals) for lag in (24, 168)}
# 각 출발 시간에 대해 1~24시간 후까지 확인된 누적 대여 건수를 저장한다.
confirmed = np.empty((len(hours), len(ids), 24), dtype=np.int16)
coverage = []
with TemporaryDirectory(prefix="cluster_demand_") as tmp:
    cached = os.environ.get("BIKE_AGGREGATE_CACHE")
    cache = Path(cached) if cached else Path(tmp)
    if not cached:
        subprocess.run([sys.executable, str(ROOT / "전처리/rental_aggregate.py"), str(cache)], check=True)
    for month, ref in verified.iterrows():
        start = hours.get_loc(pd.Timestamp(month))
        with np.load(cache / f"{month}.npz") as z:
            station_ids, counts = z["ids"], z["counts"]
        nh = counts.shape[1]
        assert nh == pd.Timestamp(month).days_in_month * 24
        assert counts.sum() == ref["집계 대여 건수"]
        assert counts[np.isin(station_ids, sample_ids)].sum() == ref["표본 50곳 대여 건수"]
        groups = group.reindex(station_ids)
        matched = groups.notna().to_numpy()
        cluster_idx = ids.get_indexer(groups[matched])
        np.add.at(totals[start:start+nh].T, cluster_idx, counts[matched])
        assigned = int(totals[start:start+nh].sum())
        assert assigned == counts[matched].sum()
        coverage.append((month, int(counts.sum()), assigned, int(counts[~matched].sum())))
        events = np.memmap(cache / f"{month}.bin", dtype="<i4", mode="r").reshape(-1, 3)
        assert len(events) == ref["필터 후 원천 행"] and np.count_nonzero(events[:,0]) == counts.sum()
        for lag, correction in late_returns.items():
            mask = (events[:,0] > 0) & (events[:,2] > events[:,1] + lag)
            delayed = events[mask, :2]
            positions = ids.get_indexer(group.reindex(delayed[:,0]))
            keep = (positions >= 0) & (delayed[:,1] + lag < len(hours))
            np.add.at(correction, (delayed[keep,1] + lag, positions[keep]), 1)
        # 반납 확인이 출발 다음 정시보다 늦은 대여만 지연 시간별로 센다.
        bins = np.zeros((26, nh, len(ids)), dtype=np.int32)
        for block_start in range(0, len(events), 250_000):
            block = events[block_start:block_start+250_000]
            delay = block[:,2] - block[:,1]
            delayed = (block[:,0] > 0) & (delay > 1)
            group_pos = ids.get_indexer(group.reindex(block[delayed,0]))
            keep = group_pos >= 0
            np.add.at(bins, (np.minimum(delay[delayed][keep],25),
                             block[delayed,1][keep] - start, group_pos[keep]), 1)
        pending = np.zeros((nh, len(ids)), dtype=np.int32)
        monthly_counts = totals[start:start+nh]
        assert monthly_counts.max() <= np.iinfo(np.int16).max
        for lag in range(24,0,-1):
            pending += bins[lag+1]
            as_known = monthly_counts - pending
            assert as_known.min() >= 0
            confirmed[start:start+nh,:,lag-1] = as_known
        del bins, block, events
demand = pd.DataFrame(totals, index=hours, columns=ids)
coverage = pd.DataFrame(coverage, columns=["월", "서울 전체", "군집 배정", "명단 외"])
assert len(coverage) == 24 and (coverage["서울 전체"] == coverage["군집 배정"] + coverage["명단 외"]).all()
annual = coverage.groupby(coverage["월"].str[:4])[["서울 전체", "군집 배정", "명단 외"]].sum()
annual["배정률(%)"] = 100 * annual["군집 배정"] / annual["서울 전체"]
display(annual.round(2))

,서울 전체,군집 배정,명단 외,배정률(%)
월,,,,
2023,43994125,42810716,1183409,97.31
2024,42688696,42219615,469081,98.90


In [3]:
import torch
from torch import nn
import math

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH, EPOCHS, SEED = 8192, 4, 42
torch.set_num_threads(4)
torch.set_float32_matmul_precision("high")
N = len(ids)
deltas = torch.arange(24,0,-1,device=DEVICE)
np.testing.assert_array_equal(confirmed[:-24,:,23], totals[:-24] - late_returns[24][24:])
weekly = np.zeros_like(totals)
weekly[168:] = totals[:-168] - late_returns[168][168:]
assert weekly.min() >= 0 and weekly.max() <= np.iinfo(np.int16).max
past = torch.as_tensor(confirmed, device=DEVICE)
week = torch.as_tensor(weekly.astype(np.int16), device=DEVICE)
actual = torch.as_tensor(totals, device=DEVICE)
weather_names = ["temperature","humidity","precipitation","windspeed"]
wx = pd.read_csv(ROOT / "분석용 데이터/Seoul_public_bicycle.csv",parse_dates=["datetime"]).set_index("datetime")
w = wx[weather_names].reindex(hours - pd.Timedelta(hours=1)).to_numpy(dtype="float32")
fixed = np.column_stack([np.eye(24)[hours.hour], np.eye(7)[hours.weekday],
                         np.eye(12)[hours.month-1], wx.holiday.loc[hours].to_numpy()]).astype("float32")
def context_at(cutoff):
    reference = w[168:cutoff-23]
    median = np.nanmedian(reference,axis=0)
    reference = np.where(np.isnan(reference),median,reference)
    mean, std = reference.mean(axis=0), reference.std(axis=0).clip(1e-6)
    values = (np.where(np.isnan(w),median,w)-mean)/std
    combined = np.column_stack([fixed,values,np.isnan(w)])
    assert np.isfinite(combined).all()
    return torch.as_tensor(combined,device=DEVICE), {"median":median.tolist(),"mean":mean.tolist(),"std":std.tolist()}

### 최근 반납량과 유사 기상 출발량

예측 정시가 09시이면 같은 군집에 08~09시 직전, 06~09시 직전 도착한 유효 대여를 각각 합산한다. 출발지는 서울 전체이며 군집 내 이동도 목적지 군집에서 한 번 센다. 이용시간·거리·중복 처리 기준은 기존 실험과 같다. 시간대 첫 경계의 불완전한 창은 앞선 168시간을 제외하는 학습·평가 규칙에 따라 사용되지 않는다.

같은 시각·요일 유형(평일 또는 주말·공휴일)·강수 유무를 가진 지난 365일 중 기온 ±5°C, 습도 ±15%p, 강수량 ±2mm, 풍속 ±2m/s 조건을 모두 만족하는 시각을 검색한다. 네 변수의 차이를 각각 허용 차이로 나누고 제곱합이 작은 순으로 최대 5개를 택한다. 모두 직전 시각의 기상 관측값으로 비교한다. 서울 기상이 공통이어서 매칭 날짜는 같지만, 평균 출발량은 각 군집 고유의 값이다. 각 과거 출발에서 24시간 뒤까지 확인된 값만 포함하며, 매칭 실패는 별도 표시한다.

In [4]:
return_cache = ROOT / "전처리/1021개 군집 시간별 반납.npz"
if not return_cache.is_file():
    subprocess.run([sys.executable, str(ROOT / "전처리/cluster_returns.py")], check=True)
with np.load(return_cache) as cache:
    np.testing.assert_array_equal(cache["ids"],ids)
    np.testing.assert_array_equal(cache["station_ids"],group.index)
    np.testing.assert_array_equal(cache["station_groups"],group)
    returned = cache["counts"]
assert returned.shape == totals.shape and returned.min() >= 0 and returned.max() <= 32767
arrivals = torch.as_tensor(returned.astype(np.int16),device=DEVICE)

limits = np.array([5.,15.,2.,2.])  # temperature, humidity, precipitation, windspeed
assert weather_names == ["temperature","humidity","precipitation","windspeed"]
complete = np.isfinite(w).all(axis=1)
wet = w[:,2] > 0
off_day = (hours.weekday >= 5) | wx.holiday.loc[hours].eq(1).to_numpy()
weather_avg = np.full(totals.shape, np.nan, dtype=np.float32)
match_count = np.zeros(len(hours),dtype="int8")
offsets = np.arange(1,366)*24
for t in range(len(hours)):
    if not complete[t]:
        continue
    prev = t-offsets[offsets<=t]
    prev = prev[complete[prev] & (off_day[prev]==off_day[t]) & (wet[prev]==wet[t])]
    differences = np.abs(w[prev]-w[t])/limits
    close = (differences <= 1).all(axis=1)
    distances = (differences[close]**2).sum(axis=1)
    selected = prev[close][np.argsort(distances,kind="stable")[:5]]
    if len(selected):
        assert (selected+24 <= t).all()
        match_count[t] = len(selected)
        weather_avg[t] = confirmed[selected,:,23].mean(axis=0)
assert np.array_equal(np.isnan(weather_avg).all(axis=1),match_count==0)
matched_avg = torch.as_tensor(weather_avg,device=DEVICE)
missing_match = torch.as_tensor(match_count==0,device=DEVICE)
print(f"군집 도착 기록 {returned.sum():,}건")
display(pd.DataFrame({"매칭 비율":(match_count>0).astype(float)},index=hours).loc["2024"]
        .resample("QS").mean().round(4))


군집 도착 기록 84,684,782건


,매칭 비율
2024-01-01,0.8489
2024-04-01,0.9734
2024-07-01,0.9665
2024-10-01,0.9176


In [5]:
VARIANTS = {
    "base": ("없음", []),
    "both": ("둘 다", ["returns_prev_1h","returns_prev_3h","weather_match_mean","weather_match_missing"]),
    "returns_only": ("반납량만", ["returns_prev_1h", "returns_prev_3h"]),
    "weather_match_only": ("유사 기상만", ["weather_match_mean", "weather_match_missing"]),
}

def batch_inputs(flat, variant):
    t = flat // N
    g = flat % N
    seq = past[t[:,None]-deltas,g[:,None],deltas-1].float().log1p()
    weekly = week[t,g].float().log1p()
    values = [weekly]
    if variant in ("returns_only","both"):
        last = arrivals[t-1,g].float()
        last_three = last + arrivals[t-2,g] + arrivals[t-3,g]
        values.extend([last.log1p(),last_three.log1p()])
    if variant in ("weather_match_only","both"):
        values.extend([matched_avg[t,g].nan_to_num().log1p(),missing_match[t].float()])
    historical = torch.stack(values,dim=1)
    return t, g, seq, historical

class ClusterNet(nn.Module):
    def __init__(self, kind, context_size, extra_size):
        super().__init__()
        self.embedding = nn.Embedding(N, 8)
        self.gru = nn.GRU(1,16,batch_first=True) if kind == "GRU" else None
        history_size = 16 if self.gru is not None else 24
        self.head = nn.Sequential(nn.Linear(history_size + context_size + 8 + 1 + extra_size, 64), nn.ReLU(),
                                  nn.Linear(64,32), nn.ReLU(), nn.Linear(32,1), nn.Softplus())

    def forward(self, seq, historical_features, context, g):
        history = seq
        if self.gru is not None:
            _, hidden = self.gru(seq.unsqueeze(-1))
            history = hidden[-1]
        features = torch.cat([history,historical_features,context,self.embedding(g)],dim=1)
        return self.head(features).squeeze(1)

print(f"학습 장치 {DEVICE}, 군집 {N:,}개")

학습 장치 cuda, 군집 1,021개


## 기존 500m 모델 복원 및 4분기 검증

날짜와 군집순서를 고정하고 저장된 설정·가중치를 복원한다. 세부 군집에서 재계산한 MAE·RMSE가 기존 CSV와 일치하는 것을 확인한 뒤 같은 시각의 500m 예측과 관측을 각각 600m 군집별로 합산한다. 합산 전 예측은 반올림하지 않는다.

In [6]:
assignment = pd.read_csv(ROOT/"따릉이 대여소 비교/실루엣 선택 군집 배정.csv").set_index("대여소번호")
np.testing.assert_array_equal(assignment.index.sort_values(),group.index.sort_values())
parents = pd.DataFrame({"500m":group,"600m":assignment["군집번호"]}).groupby("500m")["600m"]
assert parents.nunique().eq(1).all()
parent_ids = pd.Index(sorted(assignment["군집번호"].unique()))
parent_pos = parent_ids.get_indexer(parents.first().loc[ids])
assert len(parent_ids)==855 and (parent_pos>=0).all()
q4 = hours.get_loc(pd.Timestamp("2024-10-01 00:00:00"))
context, scaling = context_at(q4)
target = totals[q4:]
coarse_target = np.zeros((len(target),len(parent_ids)),dtype="int32")
np.add.at(coarse_target.T,parent_pos,target.T)
np.testing.assert_array_equal(coarse_target.sum(axis=1),target.sum(axis=1))
expected_totals = pd.read_csv(OUT/"실루엣 군집별 대여 수요.csv").set_index("군집번호")
for year in [2023,2024]:
    annual_counts = totals[hours.year==year].sum(axis=0,dtype="int64")
    aggregated = np.zeros(len(parent_ids),dtype="int64")
    np.add.at(aggregated,parent_pos,annual_counts)
    np.testing.assert_array_equal(aggregated,expected_totals.loc[parent_ids,f"{year}년 출발 건수"])


# 모형별 가중치 파일 접미어는 기존 세 노트북의 저장 형식을 따른다.
suffix = {"base":"","returns_only":"_returns_only","weather_match_only":"_weather_match_only","both":"_returns_weather"}
fine_ref = pd.read_csv(OUT/"1021개 군집 피처 개별 비교.csv").query("분기 == 4").set_index(["모델","추가 입력"])
direct = pd.read_csv(OUT/"실루엣 군집 피처별 성능.csv").query("분기 == 4").set_index(["모델","추가 입력"])
rows = []
def errors(pred, target):
    err = np.asarray(pred,dtype="float64") - target
    return np.abs(err).mean(), np.sqrt(np.square(err).mean())

for variant,(label,features) in VARIANTS.items():
    for kind in ("MLP","GRU"):
        path = OUT/f"cluster1021_{kind.lower()}{suffix[variant]}.pt"
        ck = torch.load(path,map_location="cpu",weights_only=True)
        assert pd.Timestamp(ck["cutoff"])==hours[q4] and ck["cluster_ids"]==ids.tolist()
        for key,value in scaling.items():
            np.testing.assert_allclose(ck["scaling"][key],value)
        net = ClusterNet(kind,context.shape[1],len(features)).to(DEVICE)
        net.load_state_dict(ck["weights"])
        net.eval()
        pred = np.empty(target.size,dtype="float32")
        with torch.inference_mode():
            for start in range(0,pred.size,BATCH):
                end = min(start+BATCH,pred.size)
                flat = torch.arange(q4*N+start,q4*N+end,device=DEVICE)
                t,g,seq,history = batch_inputs(flat,variant)
                pred[start:end] = net(seq,history,context[t],g).cpu().numpy()
        pred = pred.reshape(target.shape)
        ref = fine_ref.loc[(kind,label)]
        np.testing.assert_allclose(errors(pred,target),[ref.MAE,ref.RMSE],rtol=2e-5,atol=2e-5)
        coarse_pred = np.zeros(coarse_target.shape,dtype="float64")
        np.add.at(coarse_pred.T,parent_pos,pred.T)
        mae,rmse = errors(coarse_pred,coarse_target)
        scored = direct.loc[(kind,label)]
        assert scored["군집·시간 수"]==coarse_target.size
        rows.append([kind,label,coarse_target.size,mae,scored.MAE,scored.MAE-mae,
                     rmse,scored.RMSE,scored.RMSE-rmse])
        del net
same = pd.DataFrame(rows,columns=["모델","추가 입력","군집·시간 수","500m 예측 합산 MAE","600m 직접 MAE",
                                  "MAE 차이(직접-합산)","500m 예측 합산 RMSE","600m 직접 RMSE","RMSE 차이(직접-합산)"])
display(same.round(4))
same.to_csv(OUT/"동일 855개 군집 4분기 성능 비교.csv",index=False,encoding="utf-8-sig")

,모델,추가 입력,군집·시간 수,500m 예측 합산 MAE,600m 직접 MAE,MAE 차이(직접-합산),500m 예측 합산 RMSE,600m 직접 RMSE,RMSE 차이(직접-합산)
0,MLP,없음,1887840,1.9843,1.9254,-0.0589,3.4044,3.3084,-0.0960
1,GRU,없음,1887840,1.9203,1.9397,0.0194,3.2549,3.3073,0.0524
2,MLP,둘 다,1887840,1.8307,1.8891,0.0583,3.1234,3.3375,0.2140
3,GRU,둘 다,1887840,1.8343,1.8743,0.0400,3.1448,3.1785,0.0338
4,MLP,반납량만,1887840,1.9155,1.9316,0.0161,3.2854,3.3527,0.0673
5,GRU,반납량만,1887840,1.8521,1.8994,0.0473,3.1669,3.2430,0.0761
6,MLP,유사 기상만,1887840,1.8639,1.8733,0.0094,3.1908,3.2728,0.0820
7,GRU,유사 기상만,1887840,1.8782,1.8881,0.0099,3.1980,3.2102,0.0123
